# ClaimIQ - Week 6
## Data Quality: Silver Candidate to Trusted Silver and Quarantine

**ZENAIZ x BVRIT Hyderabad Data Engineering Internship**
**Project:** P19 ClaimIQ
**Notebook path:** `notebooks/04_data_quality_checks.ipynb`
**Companion files:** `src/data_quality_rules.py`, `docs/data_quality_summary.md`, `weekly_logs/week06_log.md`
**Technology:** Databricks Free Edition | Spark SQL | Delta tables

> **Week 6 job:** Execute the approved ClaimIQ rulebook **DQ01-DQ08** against every Week-5 Candidate record. Records that pass every applicable rule become **Trusted Silver**. Records that fail move to a single **`quarantine_claim_records`** table with every failure reason retained. No record is silently deleted, and Gold is only allowed to read Trusted Silver after this notebook's checks pass.

## 1. Outcome first - what will ClaimIQ produce?

Week 5 produced typed, standardised **Silver Candidate** tables for all six ClaimIQ sources. Candidate means *ready for quality assessment*; it does not yet mean trusted.

| Week-5 Candidate input | Week-6 Trusted output | Week-6 Quarantine output |
|---|---|---|
| `silver_claimiq_claims_candidate` | `trusted_silver_claimiq_claims` | `quarantine_claim_records` (filtered `entity_name = 'claims'`) |
| `silver_claimiq_policies_candidate` | `trusted_silver_claimiq_policies` | `quarantine_claim_records` (filtered `entity_name = 'policies'`) |
| `silver_claimiq_policyholders_candidate` | `trusted_silver_claimiq_policyholders` | `quarantine_claim_records` (filtered `entity_name = 'policyholders'`) |
| `silver_claimiq_products_candidate` | `trusted_silver_claimiq_products` | `quarantine_claim_records` (filtered `entity_name = 'products'`) |
| `silver_claimiq_providers_candidate` | `trusted_silver_claimiq_providers` | `quarantine_claim_records` (filtered `entity_name = 'providers'`) |
| `silver_claimiq_claim_payments_candidate` | `trusted_silver_claimiq_claim_payments` | `quarantine_claim_records` (filtered `entity_name = 'claim_payments'`) |

Unlike a per-entity quarantine table, the ClaimIQ playbook names **one shared quarantine table**, `quarantine_claim_records`, that carries every failed physical record from every entity together with its entity name, business key, failed rule IDs, severity and original payload. This keeps a single place to review, prioritise and replay failures across the whole claim lifecycle.

The non-negotiable proof for each entity is:

`Candidate rows = Trusted rows + Quarantine rows`

**Success standard:** every Candidate physical record (identified by `source_record_id`) appears exactly once on one side of the split, and every quarantined row explains why it failed.

## 2. Week 6 in one minute

Think of an insurance claims back-office intake desk:

| ClaimIQ concept | Back-office analogy | Meaning |
|---|---|---|
| Silver Candidate | claim folders waiting for the quality desk | prepared records awaiting DQ |
| DQ01-DQ08 rulebook | the documented intake checklist | approved conditions, not personal guesses |
| Trusted Silver | folders stamped "cleared for reporting" | records that passed every governing rule |
| Quarantine (`quarantine_claim_records`) | the exceptions tray | records held with clear failure reasons, never shredded |
| Diagnostic profile | a note in the margin | useful to investigate, but not a reason to reject |

> A value can be unusual without being invalid. Quarantine only when an approved ClaimIQ rule (DQ01-DQ08) fails.

## 3. What you will learn and do

By the end of this notebook, you will be able to:

1. explain the difference between profiling, diagnostic checks and governing rules;
2. implement completeness, uniqueness, reference-integrity, range, chronology, cross-field and lineage checks across six related entities;
3. use readable `CASE WHEN` statements to mark each rule `PASS` or `FAIL`;
4. keep every failure reason when one physical record breaks several rules;
5. route records into entity-specific Trusted Silver tables and a single shared `quarantine_claim_records` table;
6. reconcile counts and physical-record membership per entity;
7. explain controlled reruns and the correct-and-replay pattern;
8. explain why a reference table (policyholders, products, providers) must itself be trustworthy before a claim or policy can lean on it.

**Coding approach:** one small step at a time - prepare, check, inspect, summarise, route and prove.

## 4. Today's action journey

| Action | What you do | Evidence produced |
|---:|---|---|
| 1 | Confirm the Week-5 Candidate handoff | six Candidate tables, counts and lineage fields |
| 2 | Read the ClaimIQ DQ rulebook (DQ01-DQ08) | rule IDs, meanings and severity |
| 3 | Apply DQ01-DQ08 one rule family at a time, per entity | visible `PASS/FAIL` columns |
| 4 | Capture all failures for each row | failure count and readable reasons |
| 5 | Route each entity | six Trusted Silver Delta tables + one Quarantine Delta table |
| 6 | Prove no silent loss | count and membership reconciliation per entity |
| 7 | Test rerun and explain replay | repeat-run evidence and recovery note |

Do not jump directly to table creation. The inspection steps are where you learn to explain the result.

## 5. Three levels of quality checking

| Level | Question | ClaimIQ example | Changes route? |
|---|---|---|---|
| Profile | What values and patterns exist? | count claims by `claim_status` | No |
| Diagnostic | Does something deserve investigation? | non-zero gap between `approved_amount` and cumulative paid | No, until a threshold is approved |
| Governing rule | Does the record violate an approved condition (DQ01-DQ08)? | `loss_date` after `submission_timestamp` | Yes |

**Why this matters:** if interns convert every unusual value into a FAIL rule, valid claims may be rejected. If they only profile, invalid claims may reach Gold and Power BI. Week 6 teaches the difference.

## 6. ClaimIQ DQ rulebook - written for humans

The playbook defines exactly eight governing-rule families, DQ01-DQ08. Several families apply to more than one entity because claims, policies and payments are related. The table below is the approved rulebook; the sub-notes explain how each family is instantiated per entity in the code that follows.

| Rule ID | Record fails when... | Why it matters | Severity |
|---|---|---|---|
| `DQ01` | the entity's business key (`claim_id`, `policy_id`, `payment_id`) is null/blank, or that nonblank key occurs more than once at Trusted grain | the record cannot be identified or is double-countable | Critical |
| `DQ02` | a required `policyholder_id`, `policy_id`, `product_id` or `provider_id` reference does not resolve to its Candidate master/detail table | the claim or policy leans on a relationship that does not exist | Critical |
| `DQ03` | `loss_date` falls outside the matched policy's coverage period, or is after `submission_timestamp` | a loss cannot be covered by a policy that was not active, and cannot be reported before it happened | Major |
| `DQ04` | lifecycle timestamps do not follow `submission_timestamp <= review_timestamp <= decision_timestamp <= settlement_timestamp <= closure_timestamp` | the claim lifecycle is impossible | Major |
| `DQ05` | `requested_amount`, `approved_amount`, `reserve_amount`, `deductible_amount` or `paid_amount` is negative, or exceeds the matched policy's `coverage_limit` | monetary fields must stay inside a physically and contractually sane range | Major |
| `DQ06` | `approved_amount` exceeds `requested_amount`, or cumulative `paid_amount` exceeds `approved_amount` or the coverage limit, with no `exception_code` on file | payouts must not exceed what was asked for or covered, unless an approved exception explains it | Major |
| `DQ07` | a claim in a closed/final state is missing `outcome_code` or `closure_timestamp`, or a claim that is **not** closed already carries a `closure_timestamp` | a closed claim without an outcome cannot be reported, and an open claim cannot already be closed | Major |
| `DQ08` | a payment's `claim_id`/`provider_id` reference does not resolve, its `payment_sequence` repeats within a claim, or its `payment_status` regresses out of order | payment history must be traceable, ordered and non-contradictory | Major |

**Engineering decision - extending DQ01 to reference data:** DQ01 names `claim_id`, `policy_id` and `payment_id` explicitly. `policyholder_id`, `product_id` and `provider_id` are the business keys of the three reference/master entities. DQ02 depends on those keys being unique and present in their own tables, or the reference check is meaningless. This notebook therefore applies the same completeness-and-uniqueness **principle** as DQ01 to `policyholders`, `products` and `providers`, labelled `DQ01-PH`, `DQ01-PRD` and `DQ01-PRV` in code and evidence. This is a documented extension of an approved rule to reference data, not an invented new rule family.

The rule ID is a short, stable name: `DQ` = ClaimIQ Data Quality, and the number identifies the rule family. Suffixes (`-CLM`, `-POL`, `-PAY`, `-PH`, `-PRD`, `-PRV`) identify which entity's Candidate rows the instance of that family is evaluating. Rule IDs make code, evidence and discussions consistent.

## 7. Scenario gallery - predict before running

| Scenario | Expected route | Reason |
|---|---|---|
| Complete claim; known policyholder, policy, product; loss inside coverage; clean lifecycle; amounts in range | Trusted | every governing rule passes |
| Missing `policyholder_id` on a claim | Quarantine | `DQ02-CLM-PH` |
| Claim references a `policy_id` that does not exist in Candidate policies | Quarantine | `DQ02-CLM-POL` |
| `loss_date` before the policy's `policy_start_date` | Quarantine | `DQ03` |
| `review_timestamp` earlier than `submission_timestamp` | Quarantine | `DQ04` |
| Duplicate `claim_id` on two physical rows | both physical rows to Quarantine | `DQ01-CLM`; no approved rule chooses a winner |
| Closed claim with missing `outcome_code` and a negative payment | one Quarantine claim row and one Quarantine payment row | `DQ07` and `DQ05-PAY`; this is the Week-06 scenario from the playbook, and both rows must stay visible, never deleted |
| `approved_amount` greater than `requested_amount` with no `exception_code` | Quarantine | `DQ06` |
| Non-zero gap between `approved_amount` and cumulative paid amount, within the coverage limit | diagnostic only | unusual is not automatically invalid |

**Why keep every reason?** Fixing the first problem does not guarantee that the record is valid. One physical row stays one row; its failure summary may contain several rule IDs.

## 8. Confirm the Week-5 handoff

### 8.1 Select the working location

**Purpose:** use the same catalog and schema as the completed Week-5 notebook. Change these two settings only if your approved workspace uses another location.

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema()  AS active_schema;

**Expected result:** one row showing your intended catalog and schema. If it is wrong, stop and correct the configuration before continuing.

### 8.2 Confirm all Candidate inputs

**Purpose:** Week 6 must read the outputs of Week 5, not rebuild Silver transformations.

In [ ]:
%sql
SHOW TABLES LIKE 'silver_claimiq_*_candidate';

**Expected result:** six Candidate tables - `claims`, `policies`, `policyholders`, `products`, `providers`, `claim_payments`. Stop if any is missing; repair and reconcile Week 5 first.

### 8.3 Record the starting counts

These are the control totals we must explain at the end.

In [ ]:
%sql
SELECT 'claims' AS entity, COUNT(*) AS candidate_rows FROM silver_claimiq_claims_candidate
UNION ALL SELECT 'policies', COUNT(*) FROM silver_claimiq_policies_candidate
UNION ALL SELECT 'policyholders', COUNT(*) FROM silver_claimiq_policyholders_candidate
UNION ALL SELECT 'products', COUNT(*) FROM silver_claimiq_products_candidate
UNION ALL SELECT 'providers', COUNT(*) FROM silver_claimiq_providers_candidate
UNION ALL SELECT 'claim_payments', COUNT(*) FROM silver_claimiq_claim_payments_candidate;

**Expected result:** one real Candidate count per entity. This notebook does not invent or disclose the values.

### 8.4 Confirm physical-record lineage fields

`source_record_id` is the physical-record key carried from Bronze through Candidate for every ClaimIQ source; it is what we reconcile on. `source_system` is the lineage field that shows where the physical record originated. Both are especially important when a nonblank business key (like `claim_id`) repeats on more than one physical row.

In [ ]:
%sql
SELECT claim_id, source_record_id, source_system
FROM silver_claimiq_claims_candidate
LIMIT 5;

**Checkpoint:** explain the difference between `claim_id` (business identity) and `source_record_id` (physical-row traceability).

## 9. Learn the code pattern with one rule

The rulebook deliberately uses a familiar pattern:

```sql
CASE WHEN invalid_condition THEN 'FAIL' ELSE 'PASS' END
```

Start with missing `policyholder_id` on a claim. This cell only demonstrates the rule; it does not write a final table.

In [ ]:
%sql
SELECT claim_id, policyholder_id,
       CASE
         WHEN policyholder_id IS NULL OR trim(policyholder_id) = '' THEN 'FAIL'
         ELSE 'PASS'
       END AS policyholder_ref_present_check
FROM silver_claimiq_claims_candidate
LIMIT 20;

**Read it aloud:** "When the policyholder reference is missing or blank, mark FAIL; otherwise mark PASS."

**Expected result:** a visible check column. The sample size may or may not contain a failure; use the later summary query to measure the full table.

## 10. Build the reference-entity checks (policyholders, products, providers)

Policyholders, products and providers are small, low-change master tables. Every claim and policy leans on them, so DQ01-PH/PRD/PRV (identity) must be settled first, before we can trust any DQ02 reference check that points at them. We keep this deliberately light: identity and lineage only. Do not invent domain or range rules for these tables unless your approved contract supplies one.

### 10.1 Policyholders

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_policyholder_ids AS
SELECT policyholder_id, COUNT(*) AS occurrences
FROM silver_claimiq_policyholders_candidate
WHERE policyholder_id IS NOT NULL AND trim(policyholder_id) <> ''
GROUP BY policyholder_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policyholders_routed AS
SELECT p.*,
  CASE WHEN p.policyholder_id IS NULL OR trim(p.policyholder_id) = ''
       THEN 'FAIL' ELSE 'PASS' END AS dq01_ph_key_check,
  CASE WHEN d.policyholder_id IS NOT NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq01_ph_duplicate_check,
  CASE WHEN p.source_record_id IS NULL OR p.source_system IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS lineage_check
FROM silver_claimiq_policyholders_candidate p
LEFT JOIN duplicate_policyholder_ids d ON p.policyholder_id = d.policyholder_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policyholders_final AS
SELECT *,
  concat_ws(', ',
    CASE WHEN dq01_ph_key_check = 'FAIL' THEN 'DQ01-PH' END,
    CASE WHEN dq01_ph_duplicate_check = 'FAIL' THEN 'DQ01-PH' END,
    CASE WHEN lineage_check = 'FAIL' THEN 'DQ01-PH-LINEAGE' END
  ) AS failed_rule_ids
FROM policyholders_routed;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policyholders_final_routed AS
SELECT *,
  CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
  CASE WHEN failed_rule_ids <> '' THEN 'CRITICAL' ELSE 'NONE' END AS highest_severity
FROM policyholders_final;

**Checkpoint:** inspect a few rows. A passing policyholder has an empty failure list, `dq_status = PASS` and `highest_severity = NONE`.

### 10.2 Products

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_product_ids AS
SELECT product_id, COUNT(*) AS occurrences
FROM silver_claimiq_products_candidate
WHERE product_id IS NOT NULL AND trim(product_id) <> ''
GROUP BY product_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW products_final_routed AS
SELECT p.*,
  concat_ws(', ',
    CASE WHEN p.product_id IS NULL OR trim(p.product_id) = '' THEN 'DQ01-PRD' END,
    CASE WHEN d.product_id IS NOT NULL THEN 'DQ01-PRD' END,
    CASE WHEN p.source_record_id IS NULL OR p.source_system IS NULL THEN 'DQ01-PRD-LINEAGE' END
  ) AS failed_rule_ids
FROM silver_claimiq_products_candidate p
LEFT JOIN duplicate_product_ids d ON p.product_id = d.product_id

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW products_final AS
SELECT *,
  CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
  CASE WHEN failed_rule_ids <> '' THEN 'CRITICAL' ELSE 'NONE' END AS highest_severity
FROM products_final_routed;

**Note:** `products.csv` has only 12 rows. With a table this small, a single planted duplicate or blank key changes the routing totals visibly - inspect it directly rather than trusting only the summary count.

### 10.3 Providers

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_provider_ids AS
SELECT provider_id, COUNT(*) AS occurrences
FROM silver_claimiq_providers_candidate
WHERE provider_id IS NOT NULL AND trim(provider_id) <> ''
GROUP BY provider_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW providers_final_routed AS
SELECT p.*,
  concat_ws(', ',
    CASE WHEN p.provider_id IS NULL OR trim(p.provider_id) = '' THEN 'DQ01-PRV' END,
    CASE WHEN d.provider_id IS NOT NULL THEN 'DQ01-PRV' END,
    CASE WHEN p.source_record_id IS NULL OR p.source_system IS NULL THEN 'DQ01-PRV-LINEAGE' END
  ) AS failed_rule_ids
FROM silver_claimiq_providers_candidate p
LEFT JOIN duplicate_provider_ids d ON p.provider_id = d.provider_id

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW providers_final AS
SELECT *,
  CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
  CASE WHEN failed_rule_ids <> '' THEN 'CRITICAL' ELSE 'NONE' END AS highest_severity
FROM providers_final_routed;

**Intern question:** why do policyholders, products and providers not get a DQ02 reference check? Because they are the entities being referenced *by* claims and policies; there is nothing further upstream for them to resolve against inside ClaimIQ's contract. Their own DQ01 identity check is what makes every downstream DQ02 check meaningful.

## 11. Build Policy DQ

### 11.1 Find duplicate policy IDs

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_policy_ids AS
SELECT policy_id, COUNT(*) AS occurrences
FROM silver_claimiq_policies_candidate
WHERE policy_id IS NOT NULL AND trim(policy_id) <> ''
GROUP BY policy_id
HAVING COUNT(*) > 1;

**Expected result:** one row per repeated `policy_id`, not one row per physical policy record.

### 11.2 Apply identity, reference and range checks

The policyholder and product joins are **reference-integrity checks** (DQ02). A filled `policyholder_id`/`product_id` is not enough; that ID must exist in the Candidate reference table. `coverage_limit` and `deductible_amount` must be non-negative (DQ05 applied to policies' own monetary fields), and `policy_start_date` must not be after `policy_end_date` - this is the policy-level half of the chronology family (DQ04's intent applied to a coverage window instead of a claim lifecycle).

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policies_checked AS
SELECT pol.*,
  CASE WHEN pol.policy_id IS NULL OR trim(pol.policy_id) = ''
       THEN 'FAIL' ELSE 'PASS' END AS dq01_pol_key_check,
  CASE WHEN d.policy_id IS NOT NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq01_pol_duplicate_check,
  CASE WHEN pol.policyholder_id IS NULL OR ph.policyholder_id IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq02_policyholder_check,
  CASE WHEN pol.product_id IS NULL OR prd.product_id IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq02_product_check,
  CASE WHEN pol.coverage_limit < 0 OR pol.deductible_amount < 0 OR pol.premium_amount < 0
       THEN 'FAIL' ELSE 'PASS' END AS dq05_policy_amount_check,
  CASE WHEN pol.policy_start_date IS NOT NULL AND pol.policy_end_date IS NOT NULL
             AND pol.policy_end_date < pol.policy_start_date
       THEN 'FAIL' ELSE 'PASS' END AS dq04_policy_period_check,
  CASE WHEN pol.source_record_id IS NULL OR pol.source_system IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS lineage_check
FROM silver_claimiq_policies_candidate pol
LEFT JOIN duplicate_policy_ids d ON pol.policy_id = d.policy_id
LEFT JOIN (SELECT DISTINCT policyholder_id FROM silver_claimiq_policyholders_candidate) ph
  ON pol.policyholder_id = ph.policyholder_id
LEFT JOIN (SELECT DISTINCT product_id FROM silver_claimiq_products_candidate) prd
  ON pol.product_id = prd.product_id;

**Expected result:** every Policy Candidate row remains present once, with seven visible check columns.

### 11.3 Combine reasons and decide the route

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policies_dq AS
SELECT *,
  concat_ws(', ',
    CASE WHEN dq01_pol_key_check = 'FAIL' THEN 'DQ01-POL' END,
    CASE WHEN dq01_pol_duplicate_check = 'FAIL' THEN 'DQ01-POL' END,
    CASE WHEN dq02_policyholder_check = 'FAIL' THEN 'DQ02-POL-PH' END,
    CASE WHEN dq02_product_check = 'FAIL' THEN 'DQ02-POL-PRD' END,
    CASE WHEN dq05_policy_amount_check = 'FAIL' THEN 'DQ05-POL' END,
    CASE WHEN dq04_policy_period_check = 'FAIL' THEN 'DQ04-POL' END,
    CASE WHEN lineage_check = 'FAIL' THEN 'DQ01-POL-LINEAGE' END
  ) AS failed_rule_ids
FROM policies_checked;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW policies_routed AS
SELECT *,
  CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
  CASE WHEN dq01_pol_key_check = 'FAIL' OR dq01_pol_duplicate_check = 'FAIL'
            OR dq02_policyholder_check = 'FAIL' OR dq02_product_check = 'FAIL'
       THEN 'CRITICAL'
       WHEN dq05_policy_amount_check = 'FAIL' OR dq04_policy_period_check = 'FAIL'
            OR lineage_check = 'FAIL'
       THEN 'MAJOR' ELSE 'NONE' END AS highest_severity,
  current_timestamp() AS dq_checked_at,
  'CLAIMIQ-W06-V1' AS dq_ruleset_version
FROM policies_dq;

In [ ]:
%sql
SELECT policy_id, policyholder_id, product_id, coverage_limit,
       policy_start_date, policy_end_date, failed_rule_ids, dq_status
FROM policies_routed
LIMIT 20;

## 12. Build Claim DQ - complete worked example

Claims sit at the centre of ClaimIQ and carry the deepest rulebook coverage: DQ01, DQ02, DQ03, DQ04, DQ05, DQ06 and DQ07 all apply. This section walks through each family separately before combining them, so you can explain any single check without reading the whole SQL block at once.

### 12.1 Find duplicate claim IDs

Do not use `DISTINCT` or keep the first row. If the rulebook cannot identify the winner, every physical row carrying the repeated ID fails the uniqueness rule.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_claim_ids AS
SELECT claim_id, COUNT(*) AS occurrences
FROM silver_claimiq_claims_candidate
WHERE claim_id IS NOT NULL AND trim(claim_id) <> ''
GROUP BY claim_id
HAVING COUNT(*) > 1;

**Expected result:** one row per repeated `claim_id`, not one row per physical claim record.

### 12.2 Apply identity and reference checks (DQ01, DQ02)

`provider_id` is optional on a claim (the field dictionary marks it nullable), so DQ02's provider check only fails when a `provider_id` is present but does not resolve - a null provider is not itself a failure.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_identity_checked AS
SELECT c.*,
  CASE WHEN c.claim_id IS NULL OR trim(c.claim_id) = ''
       THEN 'FAIL' ELSE 'PASS' END AS dq01_key_check,
  CASE WHEN d.claim_id IS NOT NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq01_duplicate_check,
  CASE WHEN c.policyholder_id IS NULL OR ph.policyholder_id IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq02_policyholder_check,
  CASE WHEN c.policy_id IS NULL OR pol.policy_id IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq02_policy_check,
  CASE WHEN c.product_id IS NULL OR prd.product_id IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq02_product_check,
  CASE WHEN c.provider_id IS NOT NULL AND prv.provider_id IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq02_provider_check
FROM silver_claimiq_claims_candidate c
LEFT JOIN duplicate_claim_ids d ON c.claim_id = d.claim_id
LEFT JOIN (SELECT DISTINCT policyholder_id FROM silver_claimiq_policyholders_candidate) ph
  ON c.policyholder_id = ph.policyholder_id
LEFT JOIN (SELECT DISTINCT policy_id FROM silver_claimiq_policies_candidate) pol
  ON c.policy_id = pol.policy_id
LEFT JOIN (SELECT DISTINCT product_id FROM silver_claimiq_products_candidate) prd
  ON c.product_id = prd.product_id
LEFT JOIN (SELECT DISTINCT provider_id FROM silver_claimiq_providers_candidate) prv
  ON c.provider_id = prv.provider_id;

**Expected result:** the six visible check columns isolate identity and relationship problems without dropping the Candidate row.

**Intern question:** why join against Candidate policies/policyholders/products/providers rather than Trusted? We are evaluating each entity's Candidate set at the same pass. Referencing Trusted here would make Claim routing depend on Policy routing finishing first in a specific order, which is a hidden coupling. Document any different design explicitly and prove it does not hide row loss, exactly as the Week-5 handoff requires.

### 12.3 Resolve the matched policy for coverage-window and limit checks

DQ03, DQ05 and DQ06 all need the claim's *matched policy row* (its coverage dates and coverage limit), not just a yes/no resolution. Build that lookup once and reuse it.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claim_matched_policy AS
SELECT c.source_record_id,
       pol.policy_start_date, pol.policy_end_date, pol.coverage_limit AS policy_coverage_limit
FROM silver_claimiq_claims_candidate c
LEFT JOIN silver_claimiq_policies_candidate pol
  ON c.policy_id = pol.policy_id;

### 12.4 Apply coverage-window, chronology and amount-range checks (DQ03, DQ04, DQ05)

`DQ03` only evaluates when a policy actually resolved; an unresolved policy is already caught by `DQ02-CLM-POL` and is not double-punished here. `DQ04` compares each lifecycle timestamp against the latest known **prior** stage using `COALESCE`, so a legitimately still-open claim (with null `decision_timestamp`/`settlement_timestamp`/`closure_timestamp`) is never penalised for a stage that has not happened yet.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_range_checked AS
SELECT ic.*,
  CASE WHEN mp.policy_start_date IS NOT NULL
            AND (ic.loss_date < mp.policy_start_date OR ic.loss_date > mp.policy_end_date)
       THEN 'FAIL'
       WHEN ic.loss_date > to_date(ic.submission_timestamp)
       THEN 'FAIL'
       ELSE 'PASS' END AS dq03_coverage_window_check,
  CASE WHEN (ic.review_timestamp IS NOT NULL AND ic.review_timestamp < ic.submission_timestamp)
         OR (ic.decision_timestamp IS NOT NULL
             AND ic.decision_timestamp < COALESCE(ic.review_timestamp, ic.submission_timestamp))
         OR (ic.settlement_timestamp IS NOT NULL
             AND ic.settlement_timestamp < COALESCE(ic.decision_timestamp, ic.review_timestamp, ic.submission_timestamp))
         OR (ic.closure_timestamp IS NOT NULL
             AND ic.closure_timestamp < COALESCE(ic.settlement_timestamp, ic.decision_timestamp, ic.review_timestamp, ic.submission_timestamp))
       THEN 'FAIL' ELSE 'PASS' END AS dq04_chronology_check,
  CASE WHEN ic.requested_amount < 0 OR ic.approved_amount < 0
         OR ic.reserve_amount < 0 OR ic.deductible_amount < 0
       THEN 'FAIL'
       WHEN mp.policy_coverage_limit IS NOT NULL AND ic.requested_amount > mp.policy_coverage_limit
       THEN 'FAIL'
       ELSE 'PASS' END AS dq05_amount_range_check
FROM claims_identity_checked ic
LEFT JOIN claim_matched_policy mp ON ic.source_record_id = mp.source_record_id;

Notice the conditional chronology logic: an open claim may legitimately have null `decision_timestamp`/`settlement_timestamp`/`closure_timestamp`, so `COALESCE` always compares the current stage against the latest *known* prior stage, never against a stage that has not happened. Whether a `submitted` claim missing a `review_timestamp` after some elapsed time should itself be flagged is an SLA question, not a chronology question - profile it below rather than inventing a rule.

### 12.5 Apply the cross-field reconciliation and closure-consistency checks (DQ06, DQ07)

`DQ06` needs the claim's cumulative *Candidate* payments, aggregated by `claim_id`, filtered to `payment_status = 'PAID'` (a pending or reversed transaction should not count as money actually paid out). `DQ07` needs an agreed definition of "closed" - this notebook uses `lower(trim(claim_status)) = 'closed'` based on the Week-06 scenario in the playbook; confirm the exact label with the Section 13 profiling step below before treating it as final.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claim_cumulative_paid AS
SELECT claim_id, SUM(paid_amount) AS cumulative_paid_amount
FROM silver_claimiq_claim_payments_candidate
WHERE payment_status = 'PAID'
GROUP BY claim_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_all_checked AS
SELECT rc.*,
  CASE WHEN rc.approved_amount > rc.requested_amount AND rc.exception_code IS NULL
       THEN 'FAIL'
       WHEN mp.policy_coverage_limit IS NOT NULL AND rc.approved_amount > mp.policy_coverage_limit
            AND rc.exception_code IS NULL
       THEN 'FAIL'
       WHEN cp.cumulative_paid_amount IS NOT NULL AND cp.cumulative_paid_amount > rc.approved_amount
            AND rc.exception_code IS NULL
       THEN 'FAIL'
       WHEN mp.policy_coverage_limit IS NOT NULL AND cp.cumulative_paid_amount IS NOT NULL
            AND cp.cumulative_paid_amount > mp.policy_coverage_limit AND rc.exception_code IS NULL
       THEN 'FAIL'
       ELSE 'PASS' END AS dq06_reconciliation_check,
  CASE WHEN lower(trim(rc.claim_status)) = 'closed'
            AND (rc.outcome_code IS NULL OR rc.closure_timestamp IS NULL)
       THEN 'FAIL'
       WHEN lower(trim(rc.claim_status)) <> 'closed' AND rc.closure_timestamp IS NOT NULL
       THEN 'FAIL'
       ELSE 'PASS' END AS dq07_closure_consistency_check,
  CASE WHEN rc.source_record_id IS NULL OR rc.source_system IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS lineage_check
FROM claims_range_checked rc
LEFT JOIN claim_matched_policy mp ON rc.source_record_id = mp.source_record_id
LEFT JOIN claim_cumulative_paid cp ON rc.claim_id = cp.claim_id;

### 12.6 Inspect rule outcomes before routing

In [ ]:
%sql
SELECT claim_id, claim_status, outcome_code,
       dq01_key_check, dq01_duplicate_check,
       dq02_policyholder_check, dq02_policy_check, dq02_product_check, dq02_provider_check,
       dq03_coverage_window_check, dq04_chronology_check, dq05_amount_range_check,
       dq06_reconciliation_check, dq07_closure_consistency_check, lineage_check
FROM claims_all_checked
LIMIT 20;

**Checkpoint:** choose one displayed row and explain every check without reading the SQL.

### 12.7 Capture every failed rule and decide the route

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_dq AS
SELECT *,
  concat_ws(', ',
    CASE WHEN dq01_key_check = 'FAIL' THEN 'DQ01-CLM' END,
    CASE WHEN dq01_duplicate_check = 'FAIL' THEN 'DQ01-CLM' END,
    CASE WHEN dq02_policyholder_check = 'FAIL' THEN 'DQ02-CLM-PH' END,
    CASE WHEN dq02_policy_check = 'FAIL' THEN 'DQ02-CLM-POL' END,
    CASE WHEN dq02_product_check = 'FAIL' THEN 'DQ02-CLM-PRD' END,
    CASE WHEN dq02_provider_check = 'FAIL' THEN 'DQ02-CLM-PRV' END,
    CASE WHEN dq03_coverage_window_check = 'FAIL' THEN 'DQ03' END,
    CASE WHEN dq04_chronology_check = 'FAIL' THEN 'DQ04' END,
    CASE WHEN dq05_amount_range_check = 'FAIL' THEN 'DQ05-CLM' END,
    CASE WHEN dq06_reconciliation_check = 'FAIL' THEN 'DQ06' END,
    CASE WHEN dq07_closure_consistency_check = 'FAIL' THEN 'DQ07' END,
    CASE WHEN lineage_check = 'FAIL' THEN 'DQ01-CLM-LINEAGE' END
  ) AS failed_rule_ids
FROM claims_all_checked;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_routed AS
SELECT *,
  CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
  CASE WHEN dq01_key_check = 'FAIL' OR dq01_duplicate_check = 'FAIL'
            OR dq02_policyholder_check = 'FAIL' OR dq02_policy_check = 'FAIL'
            OR dq02_product_check = 'FAIL' OR dq02_provider_check = 'FAIL'
       THEN 'CRITICAL'
       WHEN dq03_coverage_window_check = 'FAIL' OR dq04_chronology_check = 'FAIL'
            OR dq05_amount_range_check = 'FAIL' OR dq06_reconciliation_check = 'FAIL'
            OR dq07_closure_consistency_check = 'FAIL' OR lineage_check = 'FAIL'
       THEN 'MAJOR' ELSE 'NONE' END AS highest_severity,
  current_timestamp() AS dq_checked_at,
  'CLAIMIQ-W06-V1' AS dq_ruleset_version
FROM claims_dq;

### 12.8 View genuine multi-rule failures

Instead of complex arrays, count commas: zero commas means one rule ID; one comma means at least two IDs.

In [ ]:
%sql
SELECT claim_id, claim_status, requested_amount, approved_amount,
       failed_rule_ids, highest_severity
FROM claims_routed
WHERE failed_rule_ids LIKE '%,%'
LIMIT 20;

**Expected result:** actual multi-failure examples if present. Do not publish or fabricate withheld defect totals.

## 13. Build Claim Payment DQ - the second worked example

Claim payments carry DQ01, DQ05 and DQ08, and DQ08 is the richest single family here: valid references, unique sequence and non-regressing states, all in one rule.

### 13.1 Find duplicate payment IDs

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_payment_ids AS
SELECT payment_id, COUNT(*) AS occurrences
FROM silver_claimiq_claim_payments_candidate
WHERE payment_id IS NOT NULL AND trim(payment_id) <> ''
GROUP BY payment_id
HAVING COUNT(*) > 1;

### 13.2 Find duplicate payment sequence within a claim

A `payment_sequence` is only meaningful per claim; two different claims can each have a "sequence 1". Partition the duplicate check by `claim_id`.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_payment_sequences AS
SELECT claim_id, payment_sequence, COUNT(*) AS occurrences
FROM silver_claimiq_claim_payments_candidate
WHERE claim_id IS NOT NULL AND payment_sequence IS NOT NULL
GROUP BY claim_id, payment_sequence
HAVING COUNT(*) > 1;

### 13.3 Profile payment status before writing a non-regressing rule

DQ08 requires a "non-regressing state" check, but the approved rank order of `payment_status` values is not handed to us - we must read it from the data, the same discipline the rulebook applies everywhere else.

In [ ]:
%sql
SELECT payment_status, COUNT(*) AS rows
FROM silver_claimiq_claim_payments_candidate
GROUP BY payment_status
ORDER BY rows DESC;

**Decision note:** rank the statuses you actually observe (for example `PENDING < PROCESSING < PAID`, with `REVERSED`/`FAILED` as terminal states that must not be followed by `PAID` for the same sequence) and adjust the `payment_status_rank` mapping below to match your approved list before trusting DQ08's non-regressing check. Do not invent a status that is not present in your data.

### 13.4 Resolve the matched policy per claim

DQ05 also range-checks a single payment's `paid_amount` against the coverage limit of the policy behind its claim. Payments carry `claim_id`, not `source_record_id` from the claims table, so build a claim-keyed lookup (distinct from Section 12.3's `source_record_id`-keyed lookup) before the combined check below.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claim_matched_policy_by_claim AS
SELECT c.claim_id,
       pol.coverage_limit AS policy_coverage_limit
FROM silver_claimiq_claims_candidate c
LEFT JOIN silver_claimiq_policies_candidate pol
  ON c.policy_id = pol.policy_id;

### 13.5 Apply identity, reference and non-regressing state checks (DQ01, DQ08, DQ05)

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_ranked AS
SELECT p.*,
  CASE p.payment_status
    WHEN 'PENDING' THEN 1
    WHEN 'PROCESSING' THEN 2
    WHEN 'PAID' THEN 3
    WHEN 'REVERSED' THEN 4
    WHEN 'FAILED' THEN 4
    ELSE NULL
  END AS payment_status_rank,
  LAG(p.payment_status) OVER (PARTITION BY p.claim_id ORDER BY p.payment_sequence) AS prior_status,
  LAG(CASE p.payment_status
        WHEN 'PENDING' THEN 1 WHEN 'PROCESSING' THEN 2 WHEN 'PAID' THEN 3
        WHEN 'REVERSED' THEN 4 WHEN 'FAILED' THEN 4 ELSE NULL END)
    OVER (PARTITION BY p.claim_id ORDER BY p.payment_sequence) AS prior_status_rank
FROM silver_claimiq_claim_payments_candidate p;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_checked AS
SELECT pr.*,
  CASE WHEN pr.payment_id IS NULL OR trim(pr.payment_id) = ''
       THEN 'FAIL' ELSE 'PASS' END AS dq01_pay_key_check,
  CASE WHEN d.payment_id IS NOT NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq01_pay_duplicate_check,
  CASE WHEN pr.claim_id IS NULL OR c.claim_id IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq08_claim_ref_check,
  CASE WHEN pr.provider_id IS NOT NULL AND prv.provider_id IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq08_provider_ref_check,
  CASE WHEN s.claim_id IS NOT NULL
       THEN 'FAIL' ELSE 'PASS' END AS dq08_sequence_check,
  CASE WHEN pr.prior_status_rank IS NOT NULL AND pr.payment_status_rank IS NOT NULL
            AND pr.payment_status_rank < pr.prior_status_rank
       THEN 'FAIL' ELSE 'PASS' END AS dq08_nonregressing_check,
  CASE WHEN pr.paid_amount < 0
       THEN 'FAIL'
       WHEN mp.policy_coverage_limit IS NOT NULL AND pr.paid_amount > mp.policy_coverage_limit
       THEN 'FAIL'
       ELSE 'PASS' END AS dq05_pay_amount_check,
  CASE WHEN pr.source_record_id IS NULL OR pr.source_system IS NULL
       THEN 'FAIL' ELSE 'PASS' END AS lineage_check
FROM payments_ranked pr
LEFT JOIN duplicate_payment_ids d ON pr.payment_id = d.payment_id
LEFT JOIN duplicate_payment_sequences s
  ON pr.claim_id = s.claim_id AND pr.payment_sequence = s.payment_sequence
LEFT JOIN (SELECT DISTINCT claim_id FROM silver_claimiq_claims_candidate) c
  ON pr.claim_id = c.claim_id
LEFT JOIN (SELECT DISTINCT provider_id FROM silver_claimiq_providers_candidate) prv
  ON pr.provider_id = prv.provider_id
LEFT JOIN claim_matched_policy_by_claim mp ON pr.claim_id = mp.claim_id;

### 13.6 Capture every failed rule and decide the route

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_dq AS
SELECT *,
  concat_ws(', ',
    CASE WHEN dq01_pay_key_check = 'FAIL' THEN 'DQ01-PAY' END,
    CASE WHEN dq01_pay_duplicate_check = 'FAIL' THEN 'DQ01-PAY' END,
    CASE WHEN dq08_claim_ref_check = 'FAIL' THEN 'DQ08-CLAIM-REF' END,
    CASE WHEN dq08_provider_ref_check = 'FAIL' THEN 'DQ08-PROVIDER-REF' END,
    CASE WHEN dq08_sequence_check = 'FAIL' THEN 'DQ08-SEQUENCE' END,
    CASE WHEN dq08_nonregressing_check = 'FAIL' THEN 'DQ08-NONREGRESSING' END,
    CASE WHEN dq05_pay_amount_check = 'FAIL' THEN 'DQ05-PAY' END,
    CASE WHEN lineage_check = 'FAIL' THEN 'DQ01-PAY-LINEAGE' END
  ) AS failed_rule_ids
FROM payments_checked;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_routed AS
SELECT *,
  CASE WHEN failed_rule_ids = '' THEN 'PASS' ELSE 'FAIL' END AS dq_status,
  CASE WHEN dq01_pay_key_check = 'FAIL' OR dq01_pay_duplicate_check = 'FAIL'
            OR dq08_claim_ref_check = 'FAIL' OR dq08_provider_ref_check = 'FAIL'
       THEN 'CRITICAL'
       WHEN dq08_sequence_check = 'FAIL' OR dq08_nonregressing_check = 'FAIL'
            OR dq05_pay_amount_check = 'FAIL' OR lineage_check = 'FAIL'
       THEN 'MAJOR' ELSE 'NONE' END AS highest_severity,
  current_timestamp() AS dq_checked_at,
  'CLAIMIQ-W06-V1' AS dq_ruleset_version
FROM payments_dq;

In [ ]:
%sql
SELECT payment_id, claim_id, payment_sequence, payment_status, paid_amount,
       failed_rule_ids, highest_severity
FROM payments_routed
WHERE failed_rule_ids <> ''
LIMIT 20;

**Expected result:** actual quarantined payment examples if present, including the planted negative-`paid_amount` scenario from the playbook.

## 14. Cover the wider DQ families - without inventing rules

ClaimIQ's approved rulebook is DQ01-DQ08. Other DQ families exist in general data-quality practice; understand them, but only route on them when your team's contract explicitly supports it.

| DQ family | ClaimIQ treatment this week |
|---|---|
| Completeness | keys and required references govern routing (DQ01, DQ02); other nulls are profiled |
| Uniqueness | claim, policy, payment and reference-table business keys (DQ01 and its extension) |
| Reference integrity | claim→policyholder/policy/product/provider; policy→policyholder/product; payment→claim/provider (DQ02, DQ08) |
| Domain validity | profile `claim_status`, `claim_type`, `payment_status`; approve a value list before routing further on it |
| Numeric validity | negative or over-limit monetary fields route (DQ05); other tolerances need a contract |
| Timestamp presence/order | impossible lifecycle order routes (DQ04); missing-timestamp patterns are profiled |
| Status-to-field consistency | closed/open contradiction routes (DQ07); do not invent further lifecycle rules |
| Cross-field reconciliation | approved-vs-requested and cumulative-paid-vs-approved route with an exception-code escape hatch (DQ06) |
| Effective-date/capacity/compatibility | not supported by ClaimIQ's schema this week |
| Reporting window | profile `submission_timestamp`; no approved rejection window is supplied here |
| Lineage | `source_record_id` and `source_system` govern traceability |
| Multi-rule capture/severity | implemented for every routed entity |
| Split reconciliation/rerun/replay | proved later in this notebook |

### 14.1 Profile controlled values

In [ ]:
%sql
SELECT 'claim_status' AS profile, claim_status AS value, COUNT(*) AS rows
FROM silver_claimiq_claims_candidate
GROUP BY claim_status
UNION ALL
SELECT 'claim_type', claim_type, COUNT(*)
FROM silver_claimiq_claims_candidate
GROUP BY claim_type
UNION ALL
SELECT 'payment_status', payment_status, COUNT(*)
FROM silver_claimiq_claim_payments_candidate
GROUP BY payment_status
ORDER BY profile, value;

**Decision note:** a value is not invalid merely because it is rare. Add a domain rule only when the ClaimIQ contract supplies an approved value set.

### 14.2 Profile timestamp presence and status consistency

In [ ]:
%sql
SELECT claim_status,
       CASE WHEN decision_timestamp IS NULL THEN 'decision missing' ELSE 'decision present' END AS decision_state,
       CASE WHEN closure_timestamp IS NULL THEN 'closure missing' ELSE 'closure present' END AS closure_state,
       COUNT(*) AS rows
FROM silver_claimiq_claims_candidate
GROUP BY claim_status, decision_state, closure_state
ORDER BY claim_status, decision_state, closure_state;

**Intern discussion:** what business statement would you need before declaring a `denied`-status claim with no `decision_timestamp` invalid? Write that statement as a proposed rule; do not activate it without approval.

### 14.3 Profile the approved-vs-paid reconciliation gap

In [ ]:
%sql
SELECT
  CASE WHEN cp.cumulative_paid_amount IS NULL THEN 'not comparable'
       WHEN cp.cumulative_paid_amount = c.approved_amount THEN 'matches'
       ELSE 'non-zero variance' END AS reconciliation_diagnostic,
  COUNT(*) AS rows
FROM silver_claimiq_claims_candidate c
LEFT JOIN claim_cumulative_paid cp ON c.claim_id = cp.claim_id
GROUP BY reconciliation_diagnostic;

**Key lesson:** a non-zero gap between approved and cumulative paid is useful evidence (a claim still being paid out over several instalments looks exactly like this), not by itself a DQ06 failure - DQ06 only fires when the cumulative paid *exceeds* approved with no exception code.

### 14.4 Profile the submission window

In [ ]:
%sql
SELECT MIN(submission_timestamp) AS earliest_submission,
       MAX(submission_timestamp) AS latest_submission,
       SUM(CASE WHEN submission_timestamp IS NULL THEN 1 ELSE 0 END) AS missing_submission_rows
FROM silver_claimiq_claims_candidate;

**Expected result:** the observed time span and missing count. A reporting-window routing rule needs approved start/end dates.

## 15. Summarise rule results

### 15.1 Routing totals across all six entities

In [ ]:
%sql
SELECT 'claims' AS entity, dq_status, COUNT(*) AS rows FROM claims_routed GROUP BY dq_status
UNION ALL SELECT 'policies', dq_status, COUNT(*) FROM policies_routed GROUP BY dq_status
UNION ALL SELECT 'policyholders', dq_status, COUNT(*) FROM policyholders_final_routed GROUP BY dq_status
UNION ALL SELECT 'products', dq_status, COUNT(*) FROM products_final GROUP BY dq_status
UNION ALL SELECT 'providers', dq_status, COUNT(*) FROM providers_final GROUP BY dq_status
UNION ALL SELECT 'claim_payments', dq_status, COUNT(*) FROM payments_routed GROUP BY dq_status
ORDER BY entity, dq_status;

**Expected result:** actual pass/fail totals. Record them in your evidence; do not compare them with an invented answer key.

### 15.2 Claim rule scorecard

This tells us how many physical claim rows each rule failed. One row may contribute to several rule counts.

In [ ]:
%sql
SELECT 'DQ01-CLM identity' AS rule,
       SUM(CASE WHEN dq01_key_check = 'FAIL' OR dq01_duplicate_check = 'FAIL' THEN 1 ELSE 0 END) AS failed_rows
FROM claims_routed
UNION ALL SELECT 'DQ02-CLM-* reference', SUM(CASE WHEN dq02_policyholder_check = 'FAIL' OR dq02_policy_check = 'FAIL' OR dq02_product_check = 'FAIL' OR dq02_provider_check = 'FAIL' THEN 1 ELSE 0 END) FROM claims_routed
UNION ALL SELECT 'DQ03 coverage window', SUM(CASE WHEN dq03_coverage_window_check = 'FAIL' THEN 1 ELSE 0 END) FROM claims_routed
UNION ALL SELECT 'DQ04 chronology', SUM(CASE WHEN dq04_chronology_check = 'FAIL' THEN 1 ELSE 0 END) FROM claims_routed
UNION ALL SELECT 'DQ05-CLM amount range', SUM(CASE WHEN dq05_amount_range_check = 'FAIL' THEN 1 ELSE 0 END) FROM claims_routed
UNION ALL SELECT 'DQ06 reconciliation', SUM(CASE WHEN dq06_reconciliation_check = 'FAIL' THEN 1 ELSE 0 END) FROM claims_routed
UNION ALL SELECT 'DQ07 closure consistency', SUM(CASE WHEN dq07_closure_consistency_check = 'FAIL' THEN 1 ELSE 0 END) FROM claims_routed;

**Why can rule-failure totals exceed quarantined claim rows?** Because one quarantined claim can fail several rules.

## 16. Write Trusted Silver and the shared Quarantine table

The Trusted tables retain the original Candidate columns plus the check results, rule summary, severity and ruleset metadata - intentionally transparent for learning and review. The single `quarantine_claim_records` table instead standardises every failing entity onto one shared shape: `entity_name`, `source_record_id`, `business_key`, `failed_rule_ids`, `highest_severity`, `dq_checked_at`, `dq_ruleset_version` and the full original row as JSON in `raw_payload`, so a reviewer never has to open six different tables to see what failed.

### 16.1 Trusted outputs (six tables)

In [ ]:
%sql
CREATE OR REPLACE TABLE trusted_silver_claimiq_claims USING DELTA AS
SELECT * EXCEPT (failed_rule_ids, dq_status, highest_severity, dq_checked_at, dq_ruleset_version,
                  dq01_key_check, dq01_duplicate_check, dq02_policyholder_check, dq02_policy_check,
                  dq02_product_check, dq02_provider_check, dq03_coverage_window_check,
                  dq04_chronology_check, dq05_amount_range_check, dq06_reconciliation_check,
                  dq07_closure_consistency_check, lineage_check),
       dq_status, highest_severity, dq_checked_at, dq_ruleset_version
FROM claims_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE trusted_silver_claimiq_policies USING DELTA AS
SELECT * EXCEPT (failed_rule_ids, dq_status, highest_severity, dq_checked_at, dq_ruleset_version,
                  dq01_pol_key_check, dq01_pol_duplicate_check, dq02_policyholder_check,
                  dq02_product_check, dq05_policy_amount_check, dq04_policy_period_check, lineage_check),
       dq_status, highest_severity, dq_checked_at, dq_ruleset_version
FROM policies_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE trusted_silver_claimiq_policyholders USING DELTA AS
SELECT * EXCEPT (failed_rule_ids, dq_status, highest_severity,
                  dq01_ph_key_check, dq01_ph_duplicate_check, lineage_check),
       dq_status, highest_severity
FROM policyholders_final_routed WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE trusted_silver_claimiq_products USING DELTA AS
SELECT * EXCEPT (failed_rule_ids, dq_status, highest_severity)
       , dq_status, highest_severity
FROM products_final WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE trusted_silver_claimiq_providers USING DELTA AS
SELECT * EXCEPT (failed_rule_ids, dq_status, highest_severity)
       , dq_status, highest_severity
FROM providers_final WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE trusted_silver_claimiq_claim_payments USING DELTA AS
SELECT * EXCEPT (failed_rule_ids, dq_status, highest_severity, dq_checked_at, dq_ruleset_version,
                  payment_status_rank, prior_status, prior_status_rank,
                  dq01_pay_key_check, dq01_pay_duplicate_check, dq08_claim_ref_check,
                  dq08_provider_ref_check, dq08_sequence_check, dq08_nonregressing_check,
                  dq05_pay_amount_check, lineage_check),
       dq_status, highest_severity, dq_checked_at, dq_ruleset_version
FROM payments_routed WHERE dq_status = 'PASS';

**Why `SELECT * EXCEPT (...)`?** The Candidate columns and the check/route columns share a table during inspection. Trusted Silver should look like clean business data plus a small, deliberate DQ footer (`dq_status`, `highest_severity`, `dq_checked_at`, `dq_ruleset_version`) - not a wall of intermediate `PASS`/`FAIL` columns a Gold consumer never needs. If your Spark version does not support `EXCEPT`, list the columns you want explicitly instead; do not skip this cleanup.

### 16.2 The shared quarantine table

Each entity contributes its failing rows in the same shape, so the six `SELECT`s below are unioned into one table.

In [ ]:
%sql
CREATE OR REPLACE TABLE quarantine_claim_records USING DELTA AS
SELECT 'claims' AS entity_name, source_record_id, claim_id AS business_key,
       failed_rule_ids, highest_severity, dq_checked_at, dq_ruleset_version,
       to_json(struct(*)) AS raw_payload
FROM claims_routed WHERE dq_status = 'FAIL'

UNION ALL

SELECT 'policies', source_record_id, policy_id,
       failed_rule_ids, highest_severity, dq_checked_at, dq_ruleset_version,
       to_json(struct(*))
FROM policies_routed WHERE dq_status = 'FAIL'

UNION ALL

SELECT 'policyholders', source_record_id, policyholder_id,
       failed_rule_ids, highest_severity, current_timestamp(), 'CLAIMIQ-W06-V1',
       to_json(struct(*))
FROM policyholders_final_routed WHERE dq_status = 'FAIL'

UNION ALL

SELECT 'products', source_record_id, product_id,
       failed_rule_ids, highest_severity, current_timestamp(), 'CLAIMIQ-W06-V1',
       to_json(struct(*))
FROM products_final WHERE dq_status = 'FAIL'

UNION ALL

SELECT 'providers', source_record_id, provider_id,
       failed_rule_ids, highest_severity, current_timestamp(), 'CLAIMIQ-W06-V1',
       to_json(struct(*))
FROM providers_final WHERE dq_status = 'FAIL'

UNION ALL

SELECT 'claim_payments', source_record_id, payment_id,
       failed_rule_ids, highest_severity, dq_checked_at, dq_ruleset_version,
       to_json(struct(*))
FROM payments_routed WHERE dq_status = 'FAIL';

**Expected result:** seven Delta tables - six Trusted entity tables and one shared `quarantine_claim_records`. `CREATE OR REPLACE` supports this controlled snapshot learning pattern; it is not a universal production strategy. Document your team's chosen rerun design in `docs/data_quality_summary.md`.

**Why `to_json(struct(*))` for `raw_payload`?** It captures the entire failing row, including every check column, at the moment of failure - so a reviewer or a corrected-and-replayed run always has the full original context, not just the columns this notebook happened to select.

## 17. Inspect useful evidence - without exposing unnecessary data

### 17.1 Trusted example

In [ ]:
%sql
SELECT claim_id, policyholder_id, policy_id, claim_status,
       requested_amount, approved_amount, dq_status, dq_ruleset_version
FROM trusted_silver_claimiq_claims
LIMIT 10;

### 17.2 Quarantine example, by entity and severity

In [ ]:
%sql
SELECT entity_name, business_key, failed_rule_ids, highest_severity
FROM quarantine_claim_records
ORDER BY entity_name, highest_severity
LIMIT 20;

**Evidence rule:** screenshots support the working notebook; they never replace code, counts, reconciliation or commit history. Use only fictional/synthetic ClaimIQ evidence.

## 18. Prove no silent loss

For each entity:

`Candidate rows = Trusted rows + Quarantine rows`

### 18.1 Count reconciliation

In [ ]:
%sql
SELECT 'claims' AS entity,
  (SELECT COUNT(*) FROM silver_claimiq_claims_candidate) AS candidate,
  (SELECT COUNT(*) FROM trusted_silver_claimiq_claims) AS trusted,
  (SELECT COUNT(*) FROM quarantine_claim_records WHERE entity_name = 'claims') AS quarantine
UNION ALL
SELECT 'policies',
  (SELECT COUNT(*) FROM silver_claimiq_policies_candidate),
  (SELECT COUNT(*) FROM trusted_silver_claimiq_policies),
  (SELECT COUNT(*) FROM quarantine_claim_records WHERE entity_name = 'policies')
UNION ALL
SELECT 'policyholders',
  (SELECT COUNT(*) FROM silver_claimiq_policyholders_candidate),
  (SELECT COUNT(*) FROM trusted_silver_claimiq_policyholders),
  (SELECT COUNT(*) FROM quarantine_claim_records WHERE entity_name = 'policyholders')
UNION ALL
SELECT 'products',
  (SELECT COUNT(*) FROM silver_claimiq_products_candidate),
  (SELECT COUNT(*) FROM trusted_silver_claimiq_products),
  (SELECT COUNT(*) FROM quarantine_claim_records WHERE entity_name = 'products')
UNION ALL
SELECT 'providers',
  (SELECT COUNT(*) FROM silver_claimiq_providers_candidate),
  (SELECT COUNT(*) FROM trusted_silver_claimiq_providers),
  (SELECT COUNT(*) FROM quarantine_claim_records WHERE entity_name = 'providers')
UNION ALL
SELECT 'claim_payments',
  (SELECT COUNT(*) FROM silver_claimiq_claim_payments_candidate),
  (SELECT COUNT(*) FROM trusted_silver_claimiq_claim_payments),
  (SELECT COUNT(*) FROM quarantine_claim_records WHERE entity_name = 'claim_payments');

**Pass condition:** for every row, `candidate = trusted + quarantine`.

Count equality is necessary, but it does not alone prove the same physical records were routed. The next check uses the retained `source_record_id`.

### 18.2 Simple physical-record membership proof

Every Candidate claim's `source_record_id` should appear exactly once across the two destinations. Repeat the same pattern for the other five entities before you sign off the week.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claim_route_membership AS
SELECT source_record_id, COUNT(*) AS route_occurrences
FROM (
  SELECT source_record_id FROM trusted_silver_claimiq_claims
  UNION ALL
  SELECT source_record_id FROM quarantine_claim_records WHERE entity_name = 'claims'
)
GROUP BY source_record_id;

In [ ]:
%sql
SELECT COUNT(*) AS candidate_records_not_routed_once
FROM silver_claimiq_claims_candidate c
LEFT JOIN claim_route_membership r
  ON c.source_record_id = r.source_record_id
WHERE r.route_occurrences IS NULL OR r.route_occurrences <> 1;

**Expected result:** zero. If not, stop. Investigate missing lineage, a row-multiplying join or incorrect routing before proceeding.

## 19. Controlled rerun test

1. Save the current reconciliation result.
2. Rerun the helper-view, checked-view, routed-view and seven table-write cells in order.
3. Run the reconciliation and membership checks again.

**Pass condition:** the same Candidate snapshot produces the same Trusted and Quarantine counts, with no extra physical records. `dq_checked_at` may change because a new check execution occurred; business membership must remain stable.

Do not demonstrate rerun safety by deleting outputs manually.

## 20. Correction and replay - the safe pattern

The playbook's Week-06 scenario: *a closed claim with missing outcome and a negative payment are deleted instead of retained, so Candidate no longer reconciles.* Suppose the mentor asks you to correct that quarantined claim.

### Wrong approach

- edit the source file secretly;
- update `quarantine_claim_records` directly;
- insert the record straight into a Trusted Silver table;
- remove the failed row so the counts look better.

### Correct approach

1. confirm the authoritative correction and record who approved it;
2. correct or replace the controlled upstream source using the project's governed process;
3. rerun Bronze ingestion with lineage;
4. rerun Week-5 Candidate transformation;
5. apply **all** DQ01-DQ08 rules again;
6. reconcile Candidate, Trusted and Quarantine for every affected entity;
7. record before/after evidence and commit the code/documentation change.

> A correction does not "promote" a quarantine row. It creates a newly evaluated pipeline result.

This notebook explains the replay and asks the team to prove one controlled example. It does not fabricate a corrected ClaimIQ result.

## 21. Common failures and recovery

| Problem | Why it happens | Recovery |
|---|---|---|
| Candidate table not found | Week 5 incomplete or wrong catalog/schema | return to Week 5; do not recreate Candidate here |
| Counts differ | wrong path, filter or grain | return to the source reader and reconcile before downstream work |
| Metric/visual overcounts | one-to-many join multiplication (a policy or claim joined to many payments) | aggregate at the finer grain (payments) before joining up to claim-level analysis |
| Bad record disappears | a filter or `WHERE` clause silently dropped rows instead of routing them | route the failure with raw payload and reason into `quarantine_claim_records`; re-check `Candidate = Trusted + Quarantine` |
| only first failure retained | rules were written as an `ELSE IF` chain | evaluate each rule independently, then combine reasons with `concat_ws` |
| null comparison unexpectedly passes | SQL null logic was ignored | test null explicitly with `IS NULL` |
| both duplicate rows not quarantined | code arbitrarily chose a winner | quarantine all repeated-key physical rows unless an approved survivor rule exists |
| unusual value quarantined | a diagnostic was mistaken for a governing rule | return it to profiling and seek an approved threshold |
| Candidate does not equal split | filter, join or route logic lost/multiplied rows | trace `source_record_id` and repair the earliest failing step |
| direct quarantine update | pipeline governance was bypassed | correct upstream and replay the full pipeline |
| rerun duplicates output | unstable checkpoint/batch write | use idempotent `CREATE OR REPLACE` writes and a stable checkpoint/batch id |
| real outputs pasted into reference notebook | evidence was fabricated or leaked | keep notebook outputs empty; interns execute in their own workspace |

## 22. Team-of-three ownership

| Student | Deep ownership | Backup | Must explain in mentor review |
|---|---|---|---|
| Student A | DQ framework (DQ01-DQ08 implementation) | Student B | full pipeline context and owned validation |
| Student B | quarantine/replay | Student C | full pipeline context and failure recovery |
| Student C | DQ summary/reconciliation | Student A | full pipeline context and ethical limitation |

Ownership is deep, not exclusive. All three students must run the full notebook, review the code, contribute evidence, and explain at least one engineering decision.

## 23. GitHub evidence and weekly log

### Required evidence

- rulebook (DQ01-DQ08) with ID, condition, severity and owner, including the documented DQ01 extension to reference data;
- working checks for every applicable entity rule, in `notebooks/04_data_quality_checks.ipynb` and `src/data_quality_rules.py`;
- rule-failure scorecard and a safe sample of multi-rule failure context;
- seven created Delta tables (six Trusted, one shared Quarantine) and their schemas;
- Candidate/Trusted/Quarantine count reconciliation per entity;
- physical-record membership proof per entity;
- rerun evidence;
- one documented correction-and-replay scenario;
- contribution record for Students A/B/C in `weekly_logs/week06_log.md` and `docs/data_quality_summary.md`.

### Suggested commits

- `feat: implement ClaimIQ DQ01-DQ08 routing`
- `test: prove Candidate trusted quarantine reconciliation`

### Week Log entry

Record: work completed, rule decisions, observed results, blockers, rework, evidence paths, owner/contributor details and next actions.

### AI Transparency Note

State what AI assisted with, what prompts or outputs were used, what the team independently verified in Databricks, what was changed or rejected, and which student approved the final logic. Never claim an AI-generated query was validated until the team executed and checked it.

## 24. Exit checklist

- [ ] I can explain Candidate, Trusted Silver and Quarantine in my own words.
- [ ] All six Week-5 Candidate tables exist and starting counts were recorded.
- [ ] Every governing rule (DQ01-DQ08, plus the documented DQ01 extension) has a clear ID, condition, reason and severity.
- [ ] The Week-06 scenario (closed claim, missing outcome, negative payment) is retained and explainable, not deleted.
- [ ] Wider DQ families were reviewed without inventing unsupported thresholds.
- [ ] Every rule is visible as a simple `PASS/FAIL` result.
- [ ] Multi-rule failures retain every applicable reason in one physical row.
- [ ] Six Trusted Silver Delta tables and one shared `quarantine_claim_records` table were created.
- [ ] Candidate = Trusted + Quarantine passes for every entity.
- [ ] Claim (and other entity) physical `source_record_id`s appear exactly once across the split.
- [ ] Controlled rerun evidence is recorded.
- [ ] Correction and replay is documented without direct Quarantine-to-Trusted promotion.
- [ ] GitHub commits, Week Log, AI note and Student A/B/C contributions are current.

**Week-6 completion standard:** Gold may read only from governed Trusted Silver after all exit checks pass. Gold model, KPIs, Power BI and streaming reconciliation are outside this notebook.

## 25. Viva and mentor-review questions

1. Why is Silver Candidate not yet Trusted Silver?
2. Why do both physical rows with the same `claim_id` fail DQ01?
3. What is the difference between a missing reference and an unresolved reference?
4. Why is DQ04's chronology check null-aware, using `COALESCE` against the latest known prior stage?
5. How can one Quarantine claim row contain several failure reasons?
6. Why can the sum of rule-failure counts exceed the Quarantine row count?
7. Why is a non-zero gap between `approved_amount` and cumulative paid diagnostic rather than an automatic DQ06 failure?
8. What does Candidate = Trusted + Quarantine prove, and what does it not prove?
9. Why retain `source_record_id` in both Trusted and `quarantine_claim_records`?
10. Why must a corrected record replay every rule, not just the one it originally failed?
11. Why does DQ01 get extended to `policyholders`, `products` and `providers` even though the playbook text only names `claim_id`, `policy_id` and `payment_id`?
12. What evidence shows that the Week-6 run is repeatable?

### Final boundary

**Completed here:** batch DQ01-DQ08 rules, failure context, severity, Trusted/Quarantine routing, reconciliation, rerun and replay guidance.
**Next:** governed Gold model, KPIs and reconciliation (Week 07) - only after Trusted Silver is accepted.
**Later:** controlled streaming simulation and event-time quality checks (Week 10 onward).

> Build. Prove. Present. A trusted table is not trusted because of its name; it is trusted because its rules, evidence and reconciliation are explainable.